In [1]:
# Import necessary libraries
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, roc_curve, precision_recall_curve

# 1. Choose a binary classification dataset (Breast Cancer Wisconsin)
# Load the dataset
cancer = load_breast_cancer()
X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y = pd.Series(cancer.target)

# Display a quick peek at the data
print("--- Data Shape and Head ---")
print(X.shape)
print(X.head(2))
print(f"\nTarget classes: {cancer.target_names}")

--- Data Shape and Head ---
(569, 30)
   mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
0        17.99         10.38           122.8     1001.0          0.11840   
1        20.57         17.77           132.9     1326.0          0.08474   

   mean compactness  mean concavity  mean concave points  mean symmetry  \
0           0.27760          0.3001              0.14710         0.2419   
1           0.07864          0.0869              0.07017         0.1812   

   mean fractal dimension  ...  worst radius  worst texture  worst perimeter  \
0                 0.07871  ...         25.38          17.33            184.6   
1                 0.05667  ...         24.99          23.41            158.8   

   worst area  worst smoothness  worst compactness  worst concavity  \
0      2019.0            0.1622             0.6656           0.7119   
1      1956.0            0.1238             0.1866           0.2416   

   worst concave points  worst symmetry  worst fract

In [2]:
# 2. Train/test split and standardize features
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Initialize the StandardScaler
scaler = StandardScaler()

# Fit scaler on training data and transform both sets
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("--- Split and Scaling Done ---")
print(f"Train samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")

--- Split and Scaling Done ---
Train samples: 455, Test samples: 114


In [3]:
# 3. Fit a Logistic Regression model
# Initialize and train the Logistic Regression model
log_reg = LogisticRegression(solver='liblinear', random_state=42)
log_reg.fit(X_train_scaled, y_train)

# Make predictions (class labels) and prediction probabilities
y_pred = log_reg.predict(X_test_scaled)
y_pred_proba = log_reg.predict_proba(X_test_scaled)[:, 1] # Probability of the positive class (1/benign)

print("--- Model Training and Prediction Done ---")
print("First 5 predictions (labels):", y_pred[:5])
print("First 5 prediction probabilities:", y_pred_proba[:5])

--- Model Training and Prediction Done ---
First 5 predictions (labels): [0 1 0 1 0]
First 5 prediction probabilities: [5.30434015e-08 9.99988047e-01 6.00279146e-03 5.29220660e-01
 5.76439984e-10]


## ***We use scikit-learn metrics to assess the model's performance.***

In [4]:
import matplotlib.pyplot as plt
import seaborn as sns

# 4. Evaluate with confusion matrix, precision, recall, ROC-AUC

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("--- Confusion Matrix ---")
print(cm)
# Code for plotting the CM (simplified for text output)
# plt.figure(figsize=(4,3))
# sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
# plt.xlabel('Predicted')
# plt.ylabel('Actual')
# plt.title('Confusion Matrix')
# plt.show()

# Classification Report (Precision, Recall, F1-Score)
print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=cancer.target_names))

# ROC-AUC Score
roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f"ROC-AUC Score: {roc_auc:.4f}")

--- Confusion Matrix ---
[[41  1]
 [ 1 71]]

--- Classification Report ---
              precision    recall  f1-score   support

   malignant       0.98      0.98      0.98        42
      benign       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114

ROC-AUC Score: 0.9957


In [7]:
# 5. Tune threshold (Example: prioritize high Recall for the Malignant class (0))

# Get precision and recall for various thresholds
precision, recall, thresholds = precision_recall_curve(y_test, y_pred_proba)

# Find the threshold that gives a desired balance, e.g., where Precision is at least 0.90 for the positive class (benign)
# Note: For this problem, we might care more about high Recall for the *negative* class (Malignant/0)
# A simple example: Let's see the performance at a higher threshold of 0.7 instead of the default 0.5

custom_threshold = 0.7
y_pred_custom = (y_pred_proba >= custom_threshold).astype(int)

print(f"\n--- Evaluation with Custom Threshold ({custom_threshold}) ---")
print(confusion_matrix(y_test, y_pred_custom))
print(classification_report(y_test, y_pred_custom, target_names=cancer.target_names))


--- Evaluation with Custom Threshold (0.7) ---
[[41  1]
 [ 6 66]]
              precision    recall  f1-score   support

   malignant       0.87      0.98      0.92        42
      benign       0.99      0.92      0.95        72

    accuracy                           0.94       114
   macro avg       0.93      0.95      0.94       114
weighted avg       0.94      0.94      0.94       114

